# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
!pip -q install transformers sentencepiece torch


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


In [3]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [4]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
C:\Users\najaf\anaconda3\envs\lab7\Lib\site-packages\transformers\generation\utils.py:1955: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [6]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

1. Sentiment classification    -->    Encoder-only

2. Machine translation         -->    Encoder-decoder

3. Open-ended text generation  -->    Decoder-only

4. Named Entity Recognition    -->    Encoder-only

5. Text summarization          -->    Encoder-decoder

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

for s in sentences:
    r = classifier(s)[0]
    print(f"{s}\n   Label: {r['label']}   Score: {r['score']:.4f}\n")

The movie was excellent.
   Label: POSITIVE   Score: 0.9999

The lecture was difficult to understand.
   Label: NEGATIVE   Score: 0.9997

I enjoyed learning about transformers.
   Label: POSITIVE   Score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand, interpret, and generate human language. It is used in applications such as "
    "machine translation, chatbots, sentiment analysis, and search engines. Modern NLP systems "
    "are mostly built on transformer models, which learn patterns in language from huge amounts of text."
)

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
summary_ids = model.generate(**inputs, max_new_tokens=60, num_beams=4, no_repeat_ngram_size=3, early_stopping=True,)

print("Summary:")
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

Summary:
Use natural language processing.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:** A decoder-only model is suitable for a chatbot because it is trained to predict the next token given all the previous tokens, which is exactly what a conversation is: the user's messages and the earlier replies form the context, and the model continues the text one token at a time. Its causal (left-to-right) attention lets it generate fluent, open-ended responses of any length. Because the prompt and the answer live in the same sequence, the model can follow instructions, remember earlier turns, and be fine-tuned to behave like an assistant. It does not need a separate encoder, which makes it simple to scale to very large sizes.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [9]:
generator = pipeline("text-generation", model="distilgpt2")
prompt = "Natural language processing is"

# With a fixed seed -> same output every run
print("=== With set_seed(42) ===")
for i in range(3):
    set_seed(42)
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9)
    print(f"Run {i+1}: {out[0]['generated_text']}\n")

# Without a seed -> different output every run
print("=== Without set_seed ===")
for i in range(3):
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9)
    print(f"Run {i+1}: {out[0]['generated_text']}\n")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== With set_seed(42) ===


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found that most languages are languages



...

Run 3: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found that most languages are languages

=== Without set_seed ===


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Natural language processing is an area of interest for both of us and the other species.



[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Natural language processing is a fundamental part of the science and culture of the United States. As a result, our nation› has witnessed a profound increase in education, economic

Run 3: Natural language processing is a means of processing a language used by people in countries with no social networks. For the majority of people speaking in one language, language processing can be



**Answer:** 
The outputs differ because, when `do_sample=True`, the model does not always pick the single most likely next token. It computes a probability distribution over the vocabulary and **randomly samples** from it, so each run can pick different tokens, and early differences snowball into very different continuations.